# Notebook 03 — SIMD vs Scalar Paths

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 01 mapped input distributions.  
Notebook 02 introduced cache and branching proxy structure.  
Notebook 03 adds an execution-path layer:

- scalar pathways
- SIMD-friendly pathways
- branch pressure
- locality / reuse
- distribution-dependent acceleration

Constraint view:
> SIMD is not automatically “better.” It helps when data structure, algorithm path, and hardware constraints align.

## Goals

1. Load or generate structural metrics from Notebook 02.
2. Create a transparent synthetic execution-path model.
3. Estimate relative scalar vs SIMD suitability by distribution.
4. Produce figures:
   - SIMD suitability score
   - scalar vs SIMD estimated throughput
   - speedup vs branch pressure
   - execution-path phase map
5. Export CSV, JSON, Markdown report, and PNG outputs.

This notebook is a modeling bridge, not a replacement for real benchmarks.
Later notebooks can overlay upstream benchmark results and hardware counters.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 02 metrics if available

If Notebook 02 outputs are unavailable, this notebook uses a fallback structural table matching the same metric vocabulary.

In [ ]:
metrics_path = RESULTS_DIR / "notebook02_cache_branching_metrics.csv"

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print("Loaded:", metrics_path)
else:
    print("Notebook 02 metrics not found; using fallback structural metrics.")
    df = pd.DataFrame([
        {
            "name": "low_entropy_repeating",
            "digit_length_entropy": 0.0,
            "digit_length_transition_rate": 0.0,
            "locality_small_delta_ratio": 1.0,
            "cache_window_reuse_proxy": 0.94,
            "branch_pressure_score": 0.01,
            "delta_abs_mean": 1.5,
            "repetition_ratio": 0.99996,
        },
        {
            "name": "sequential_ids",
            "digit_length_entropy": 0.52,
            "digit_length_transition_rate": 0.0,
            "locality_small_delta_ratio": 1.0,
            "cache_window_reuse_proxy": 0.0,
            "branch_pressure_score": 0.20,
            "delta_abs_mean": 1.0,
            "repetition_ratio": 0.0,
        },
        {
            "name": "uniform_32bit",
            "digit_length_entropy": 0.90,
            "digit_length_transition_rate": 0.37,
            "locality_small_delta_ratio": 0.0,
            "cache_window_reuse_proxy": 0.0,
            "branch_pressure_score": 0.72,
            "delta_abs_mean": 1.4e9,
            "repetition_ratio": 0.00001,
        },
        {
            "name": "zipfian_smallints",
            "digit_length_entropy": 2.42,
            "digit_length_transition_rate": 0.74,
            "locality_small_delta_ratio": 0.27,
            "cache_window_reuse_proxy": 0.35,
            "branch_pressure_score": 0.72,
            "delta_abs_mean": 7.0e7,
            "repetition_ratio": 0.80,
        },
        {
            "name": "clustered_ranges",
            "digit_length_entropy": 1.25,
            "digit_length_transition_rate": 0.55,
            "locality_small_delta_ratio": 0.0,
            "cache_window_reuse_proxy": 0.01,
            "branch_pressure_score": 0.79,
            "delta_abs_mean": 5.0e4,
            "repetition_ratio": 0.98,
        },
    ])

df

## Synthetic SIMD / scalar execution-path model

This simple model is meant to be interpretable:

- SIMD likes regular, wide, predictable work.
- Scalar paths can remain competitive when branching, irregular lengths, or small repeated values dominate.
- Locality and reuse can help both, but often help scalar/simple paths more.
- Branch pressure penalizes SIMD suitability.

The output is a **relative score**, not an absolute benchmark claim.

In [ ]:
work = df.copy()

# Normalize helper
def norm01(series):
    s = pd.Series(series).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

work["delta_scale"] = norm01(np.log10(work["delta_abs_mean"].astype(float) + 1.0))
work["digit_entropy_scale"] = norm01(work["digit_length_entropy"])
work["branch_pressure_norm"] = norm01(work["branch_pressure_score"])
work["locality_norm"] = work["locality_small_delta_ratio"].astype(float)
work["reuse_norm"] = work["cache_window_reuse_proxy"].astype(float)

# SIMD suitability: favors broad regular work and lower branch pressure.
work["simd_suitability"] = (
    0.35 * work["delta_scale"] +
    0.25 * work["digit_entropy_scale"] +
    0.25 * (1.0 - work["branch_pressure_norm"]) +
    0.15 * (1.0 - work["reuse_norm"])
).clip(0, 1)

# Scalar suitability: favors locality, reuse, and lower branch pressure.
work["scalar_suitability"] = (
    0.35 * work["locality_norm"] +
    0.30 * work["reuse_norm"] +
    0.25 * (1.0 - work["branch_pressure_norm"]) +
    0.10 * (1.0 - work["digit_entropy_scale"])
).clip(0, 1)

# Toy throughput model: arbitrary units for visualization.
base_scalar = 1.0
base_simd = 1.0

work["estimated_scalar_throughput"] = (
    base_scalar
    * (1.0 + 0.8 * work["scalar_suitability"])
    * (1.0 - 0.35 * work["branch_pressure_norm"])
)

work["estimated_simd_throughput"] = (
    base_simd
    * (1.0 + 1.8 * work["simd_suitability"])
    * (1.0 - 0.55 * work["branch_pressure_norm"])
)

work["estimated_speedup_simd_over_scalar"] = (
    work["estimated_simd_throughput"] / work["estimated_scalar_throughput"]
)

work = work.sort_values("estimated_speedup_simd_over_scalar", ascending=False)
work

## Export model table

In [ ]:
csv_path = RESULTS_DIR / "notebook03_simd_scalar_path_metrics.csv"
json_path = RESULTS_DIR / "notebook03_simd_scalar_path_metrics.json"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — SIMD suitability by distribution

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook03_simd_suitability.png"

plot_df = work.sort_values("simd_suitability")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["name"], plot_df["simd_suitability"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("SIMD suitability score")
plt.title("Execution-Path Structure: SIMD Suitability")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Estimated scalar vs SIMD throughput

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook03_scalar_vs_simd_throughput.png"

plot_df = work.sort_values("estimated_speedup_simd_over_scalar")
x = np.arange(len(plot_df))
width = 0.38

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, plot_df["estimated_scalar_throughput"], width, label="scalar")
plt.bar(x + width/2, plot_df["estimated_simd_throughput"], width, label="SIMD")
plt.xticks(x, plot_df["name"], rotation=45, ha="right")
plt.ylabel("Estimated relative throughput")
plt.title("Toy Model: Scalar vs SIMD Relative Throughput")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Speedup vs branch pressure

This figure asks whether predicted SIMD advantage survives branch pressure.

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook03_speedup_vs_branch_pressure.png"

plt.figure(figsize=(8, 5))
plt.scatter(work["branch_pressure_score"], work["estimated_speedup_simd_over_scalar"])
for _, row in work.iterrows():
    plt.annotate(row["name"], (row["branch_pressure_score"], row["estimated_speedup_simd_over_scalar"]), fontsize=8)
plt.xlabel("Branch pressure proxy")
plt.ylabel("Estimated SIMD / scalar speedup")
plt.title("Execution-Path Structure: Speedup vs Branch Pressure")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Execution-path phase map

A simple phase map: scalar suitability vs SIMD suitability.

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook03_execution_path_phase_map.png"

plt.figure(figsize=(8, 6))
plt.scatter(work["scalar_suitability"], work["simd_suitability"])
for _, row in work.iterrows():
    plt.annotate(row["name"], (row["scalar_suitability"], row["simd_suitability"]), fontsize=8)
plt.xlabel("Scalar suitability")
plt.ylabel("SIMD suitability")
plt.title("Execution-Path Phase Map")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_03_simd_vs_scalar_paths.md"

summary_cols = [
    "name",
    "simd_suitability",
    "scalar_suitability",
    "branch_pressure_score",
    "estimated_scalar_throughput",
    "estimated_simd_throughput",
    "estimated_speedup_simd_over_scalar",
]

lines = [
    "# Report 03 — SIMD vs Scalar Paths",
    "",
    "This report adds an execution-path interpretation layer for integer serialization.",
    "",
    "Constraint view:",
    "> SIMD helps when data structure, algorithm path, and hardware constraints align.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    "",
    "## Execution-path summary",
    "",
    work[summary_cols].to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- SIMD suitability is distribution-dependent, not automatic.",
    "- Branch pressure can reduce SIMD advantage even when wide work exists.",
    "- Scalar pathways remain meaningful for low-entropy, high-reuse, or highly local distributions.",
    "- This notebook creates a modeling layer that later benchmark results can confirm, correct, or falsify.",
    "",
    "## Next step",
    "",
    "Notebook 04 should combine distribution structure, cache/branching proxies, and execution-path metrics into constraint phase maps.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook03_simd_scalar_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook03_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_03_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))